# Session 13: Virtual Private Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-13-virtual-private-networks.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-13_ipsec_concepts.py`

In [1]:
"""Session 13 - The security building blocks of a VPN, shown with real primitives.

1. Diffie-Hellman: two gateways agree on a shared secret over an open network
   (IKE does this with standard groups; here the 2048-bit RFC 3526 group 14).
2. Key derivation: turn the shared secret into separate keys (HKDF-style, HMAC-SHA256).
3. Integrity and authenticity: an HMAC tag proves a packet was not changed.
4. Anti-replay: a sequence-number window rejects copied packets.
Educational only: real VPNs use vetted implementations (IOS, strongSwan, WireGuard).
Run:  python session-13_ipsec_concepts.py
"""
import hashlib
import hmac
import secrets

P = int("FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD129024E088A67CC74020BBEA63B139B22514A08798E3404DD"
        "EF9519B3CD3A431B302B0A6DF25F14374FE1356D6D51C245E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7ED"
        "EE386BFB5A899FA5AE9F24117C4B1FE649286651ECE45B3DC2007CB8A163BF0598DA48361C55D39A69163FA8FD24CF5F"
        "83655D23DCA3AD961C62F356208552BB9ED529077096966D670C354E4ABC9804F1746C08CA18217C32905E462E36CE3B"
        "E39E772C180E86039B2783A2EC07A28FB5C55DF06F4C52C9DE2BCBF6955817183995497CEA956AE515D2261898FA0510"
        "15728E5A8AACAA68FFFFFFFFFFFFFFFF", 16)          # RFC 3526 MODP group 14 (2048-bit)
G = 2


def hkdf(secret: bytes, info: bytes, length=32) -> bytes:
    prk = hmac.new(b"lab-salt", secret, hashlib.sha256).digest()
    return hmac.new(prk, info + b"\x01", hashlib.sha256).digest()[:length]


class ReplayWindow:
    def __init__(self, size=64):
        self.size, self.highest, self.seen = size, 0, set()

    def accept(self, seq):
        if seq in self.seen or seq <= self.highest - self.size:
            return False
        self.seen.add(seq); self.highest = max(self.highest, seq)
        return True


if __name__ == "__main__":
    a_priv, b_priv = secrets.randbelow(P - 3) + 2, secrets.randbelow(P - 3) + 2
    a_pub, b_pub = pow(G, a_priv, P), pow(G, b_priv, P)
    print("1) Diffie-Hellman over an untrusted network (group 14, 2048-bit)")
    print(f"   Karachi gateway sends public value ({a_pub.bit_length()} bits)  {hex(a_pub)[:22]}...")
    print(f"   Lahore  gateway sends public value ({b_pub.bit_length()} bits)  {hex(b_pub)[:22]}...")
    s_a, s_b = pow(b_pub, a_priv, P), pow(a_pub, b_priv, P)
    print(f"   both computed the same secret: {s_a == s_b} (an eavesdropper saw only the public values)")

    shared = s_a.to_bytes(256, "big")
    k_enc, k_auth = hkdf(shared, b"encryption"), hkdf(shared, b"integrity")
    print("\n2) Separate keys derived from the shared secret")
    print(f"   encryption key length {len(k_enc) * 8} bits, integrity key length {len(k_auth) * 8} bits, "
          f"different: {k_enc != k_auth}")

    print("\n3) Integrity: HMAC-SHA256 tag on each packet")
    pkt = b"seq=1001|src=10.1.0.5|dst=10.2.0.9|payload=grade-sheet.xlsx"
    tag = hmac.new(k_auth, pkt, hashlib.sha256).digest()
    tampered = pkt.replace(b"10.2.0.9", b"10.66.6.6")
    for label, p in (("original", pkt), ("tampered in transit", tampered)):
        ok = hmac.compare_digest(tag, hmac.new(k_auth, p, hashlib.sha256).digest())
        print(f"   {label:20} -> {'ACCEPT' if ok else 'DROP (ICV mismatch)'}")

    print("\n4) Anti-replay window (64 packets)")
    w = ReplayWindow()
    for seq in (1001, 1002, 1004, 1003, 1002, 900):
        print(f"   seq {seq}: {'accept' if w.accept(seq) else 'REJECT (replayed or too old)'}")

1) Diffie-Hellman over an untrusted network (group 14, 2048-bit)
   Karachi gateway sends public value (2046 bits)  0x2e1ff16d64ea68441fa1...
   Lahore  gateway sends public value (2045 bits)  0x14828df06638c2aca152...
   both computed the same secret: True (an eavesdropper saw only the public values)

2) Separate keys derived from the shared secret
   encryption key length 256 bits, integrity key length 256 bits, different: True

3) Integrity: HMAC-SHA256 tag on each packet
   original             -> ACCEPT
   tampered in transit  -> DROP (ICV mismatch)

4) Anti-replay window (64 packets)
   seq 1001: accept
   seq 1002: accept
   seq 1004: accept
   seq 1003: accept
   seq 1002: REJECT (replayed or too old)
   seq 900: REJECT (replayed or too old)


## `session-13_vpn_overhead.py`

In [2]:
"""Session 13 - VPN tunnels cost bytes: overhead, MTU and TCP MSS.

Every tunnel wraps the original packet in new headers. If the result is
bigger than the link MTU (1500 on Ethernet) it must be fragmented or dropped,
which is the classic "VPN is up but some websites hang" problem. The fix is
to lower the tunnel MTU and clamp the TCP MSS. Header sizes below are typical
values for the listed options.
Run:  python session-13_vpn_overhead.py
"""

TUNNELS = {
    # name: list of (header, bytes)
    "GRE over IPv4": [("new IPv4 header", 20), ("GRE header", 4)],
    "IPsec ESP tunnel, AES-CBC + SHA-256": [("new IPv4 header", 20), ("ESP header (SPI+seq)", 8),
                                            ("IV", 16), ("padding (worst case)", 15),
                                            ("pad length + next header", 2), ("ICV (integrity tag)", 16)],
    "IPsec ESP tunnel, AES-GCM": [("new IPv4 header", 20), ("ESP header", 8), ("IV", 8),
                                  ("padding (worst case)", 3), ("pad length + next header", 2),
                                  ("ICV", 16)],
    "GRE inside IPsec (transport mode), AES-GCM": [("IPv4 header", 20), ("ESP header", 8), ("IV", 8),
                                                   ("padding", 3), ("pad length + next header", 2),
                                                   ("ICV", 16), ("GRE", 4)],
    "IPsec NAT-T (UDP 4500), AES-GCM": [("new IPv4 header", 20), ("UDP header", 8), ("ESP header", 8),
                                        ("IV", 8), ("padding", 3), ("pad length + next header", 2),
                                        ("ICV", 16)],
}

if __name__ == "__main__":
    LINK_MTU = 1500
    print(f"Link MTU {LINK_MTU} bytes. Inner TCP/IPv4 headers = 40 bytes.\n")
    print(f"{'Tunnel type':45} {'Overhead':>8} {'Tunnel MTU':>11} {'TCP MSS':>8}")
    for name, hdrs in TUNNELS.items():
        ovh = sum(b for _, b in hdrs)
        mtu = LINK_MTU - ovh
        print(f"{name:45} {ovh:>7}B {mtu:>10}B {mtu - 40:>7}B")
    print("\nWhat happens to a full-size 1500-byte packet in an AES-GCM ESP tunnel:")
    size = 1500 + sum(b for _, b in TUNNELS["IPsec ESP tunnel, AES-GCM"])
    print(f"  encrypted size {size} B > {LINK_MTU} B -> fragmented (slow) or dropped if DF bit is set")
    print("  Fix on the tunnel interface:  ip mtu 1400   and   ip tcp adjust-mss 1360")
    print("\nEfficiency for small VoIP packets (60 bytes) vs large file packets (1400 bytes):")
    for payload in (60, 1400):
        ovh = sum(b for _, b in TUNNELS["IPsec ESP tunnel, AES-GCM"])
        print(f"  {payload:>5} B payload: {payload / (payload + ovh) * 100:5.1f}% of the bytes on the wire are useful")

Link MTU 1500 bytes. Inner TCP/IPv4 headers = 40 bytes.

Tunnel type                                   Overhead  Tunnel MTU  TCP MSS
GRE over IPv4                                      24B       1476B    1436B
IPsec ESP tunnel, AES-CBC + SHA-256                77B       1423B    1383B
IPsec ESP tunnel, AES-GCM                          57B       1443B    1403B
GRE inside IPsec (transport mode), AES-GCM         61B       1439B    1399B
IPsec NAT-T (UDP 4500), AES-GCM                    65B       1435B    1395B

What happens to a full-size 1500-byte packet in an AES-GCM ESP tunnel:
  encrypted size 1557 B > 1500 B -> fragmented (slow) or dropped if DF bit is set
  Fix on the tunnel interface:  ip mtu 1400   and   ip tcp adjust-mss 1360

Efficiency for small VoIP packets (60 bytes) vs large file packets (1400 bytes):
     60 B payload:  51.3% of the bytes on the wire are useful
   1400 B payload:  96.1% of the bytes on the wire are useful
